# Lecture 8.4 — Guardrails at the LLM Boundary

**Design Patterns:** Guardrails & Policy Enforcement (P1) + Conditional Skipping of Steps (P6)  
**Callback used:** `before_agent_callback`  
**Applied to:** `budget_optimizer_workflow` (the SequentialAgent)

In this lecture we upgrade the Lecture 8.3 notebook with an active guardrail that stops the entire workflow before any agent runs if the topic is unsafe.

The guardrail uses an **LLM-as-judge** — a second Gemini call that semantically evaluates the event topic before any planning work begins. This is far more robust than a keyword list: it catches paraphrases, euphemisms, and novel phrasing that no static list would ever anticipate.

---
**Changes from Lecture 8.3 (the complete diff):**
1. New cell: Model configuration — `AGENT_MODEL` and `JUDGE_MODEL` constants
2. New constants: `SAFETY_VERDICT_PROMPT`, `SAFETY_REASON_PROMPT`, `REFUSAL_CONTENT`
3. New function: `guardrail_before_workflow` — a two-stage LLM judge
4. One new keyword argument on `budget_optimizer_workflow`: `before_agent_callback=guardrail_before_workflow`
5. Two execution cells — one clean topic (workflow runs), one blocked topic (cancelled instantly)

---
**Expected output — clean topic:**
```
[SAFETY JUDGE] Evaluating topic: '50 person AI event in New York'
[SAFETY JUDGE] Verdict         : NO
  └─ ✅ SAFE — starting workflow.
```
**Expected output — blocked topic:**
```
[SAFETY JUDGE] Evaluating topic: 'weapons convention'
[SAFETY JUDGE] Verdict         : YES
[SAFETY JUDGE] Generating refusal explanation...
[SAFETY JUDGE] Reason          : Thank you for reaching out. Unfortunately...
  └─ 🚫 BLOCKED — high-risk topic. Workflow cancelled.
```

## ⚙️ 1. Setup: Install Libraries

Pinning the version ensures our code will always work as expected.

In [ ]:
!pip install google-adk==1.29.0 -q

## 🔑 2. Authentication: Configure Your API Key

In [ ]:
import os
from getpass import getpass

api_key = getpass('Enter your Google API Key: ')
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured successfully!")

## 🤖 3. Model Configuration

Define the model names once here. To upgrade to a newer model in the future,
change these two constants — nothing else in the notebook needs to touch.

In [ ]:
# ── Model Configuration ───────────────────────────────────────────────────────
# Change these two constants to swap models across the entire notebook.
# To upgrade to a newer model in the future, update AGENT_MODEL and JUDGE_MODEL here.

AGENT_MODEL = "gemini-2.5-flash"  # used by all workflow agents
JUDGE_MODEL = "gemini-2.5-flash"   # used by the safety judge


## 🪝 3. [CARRIED OVER from 8.3] Define the Observability Callbacks

These two callbacks are unchanged from Lecture 8.3.  
They fire at the **agent** boundary (entry and exit of the whole workflow).  
The new callback in the next cell fires at the **model** boundary — a different, deeper hook.

In [ ]:
from datetime import datetime
from typing import Optional

from google.adk.agents.callback_context import CallbackContext
from google.genai import types

# A module-level variable so log_agent_exit can calculate elapsed time.
_workflow_start_time: datetime = None


def log_agent_entry(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    before_agent_callback for spending_proposer_agent.

    Fires once, right before the first LLM call in the entire workflow.
    Records the start time and prints a structured ENTRY log line.

    Returns None — the agent proceeds normally. Nothing is skipped.
    """
    global _workflow_start_time
    _workflow_start_time = datetime.now()  # Capture start time for later

    # --- Read from CallbackContext ---
    agent_name    = callback_context.agent_name      # e.g. 'spending_proposer_agent'
    invocation_id = callback_context.invocation_id   # unique UUID for this run
    state_keys    = list(callback_context.state.to_dict().keys())  # what's in memory so far
    timestamp     = _workflow_start_time.strftime("%H:%M:%S")

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[ENTRY] {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        state_keys : {state_keys}")
    print("="*60)

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — proceed with the agent as normal.'
    return None


def log_agent_exit(callback_context: CallbackContext) -> Optional[types.Content]:
    """
    after_agent_callback for plan_retriever_agent.

    Fires once, right after the last agent in the workflow completes.
    Calculates total elapsed time and prints a structured EXIT log line.

    Returns None — the agent's output is used unchanged. Nothing is replaced.
    """
    now        = datetime.now()
    agent_name = callback_context.agent_name
    invocation_id = callback_context.invocation_id
    state_keys = list(callback_context.state.to_dict().keys())
    timestamp  = now.strftime("%H:%M:%S")

    # Calculate duration only if log_agent_entry ran first
    if _workflow_start_time is not None:
        elapsed = (now - _workflow_start_time).seconds
        duration_str = f"{elapsed}s"
    else:
        duration_str = "n/a"

    # --- Structured log line ---
    print("\n" + "="*60)
    print(f"[EXIT]  {agent_name}")
    print(f"        inv        : {invocation_id}")
    print(f"        timestamp  : {timestamp}")
    print(f"        duration   : {duration_str}")
    print(f"        state_keys : {state_keys}")
    print("="*60 + "\n")

    # IMPORTANT: returning None tells the ADK framework
    # 'I'm done observing — use the agent's real output as-is.'
    return None

## 🛡️ 4. [NEW] Define the LLM-as-Judge Guardrail

This is the **only new code in this lecture**.

### Why `before_agent_callback` on the SequentialAgent?

| Where | Callback | Blocks | Extra plumbing |
|---|---|---|---|
| On `spending_proposer_agent` | `before_model_callback` | One LLM call — SequentialAgent keeps running | State flags + skip callbacks on every downstream agent |
| On `spending_proposer_agent` | `before_agent_callback` | That agent's run — SequentialAgent keeps running | Callback list to coexist with `log_agent_entry` |
| **On `budget_optimizer_workflow`** | **`before_agent_callback`** | **Entire workflow — no sub-agent ever starts** | **Nothing** |

Placing the guardrail on the outermost `SequentialAgent` is the cleanest position. When `before_agent_callback` returns `Content`, the agent's `_run_async_impl` is skipped entirely — meaning `spending_proposer_agent`, `budget_refinement_loop`, and `plan_retriever_agent` never run at all.

### Why an LLM judge instead of a keyword list?

A keyword list misses paraphrases. `"small arms expo"`, `"underground casino night"`, and `"recreational pharmaceutical summit"` all bypass any static word list. The LLM judge understands **intent and context**, not just surface words.

### Two-stage design

- **Stage 1** — fast binary verdict (`YES`/`NO`) on every request  
- **Stage 2** — rich, topic-specific refusal explanation, only called when Stage 1 returns `YES`

Clean topics pay for exactly one judge call. Blocked topics get a tailored explanation rather than a generic fallback string.

### Return value contract
- Return `None` → topic is safe, workflow runs normally  
- Return `Content` → entire workflow cancelled, returned content shown to user

In [ ]:
# ============================================================
# Lecture 8.4 — Upgraded Guardrail: LLM-as-Judge
# Design Patterns: Guardrails & Policy Enforcement (P1)
#                  Conditional Skipping of Steps (P6)
# ============================================================



# Initialise a direct Gemini client for the safety judge.
# This is a raw API call — completely separate from the ADK runner.


# ── Judge prompts ─────────────────────────────────────────────────────────────
# Two-stage design:
#   Prompt 1 — binary verdict (YES/NO). Fast, cheap, used on every request.
#   Prompt 2 — human-readable explanation. Only called when verdict is YES,
#              so clean topics pay no extra cost.



## 🛠️ 5. Define Workflow Tools

Unchanged from Section 5 and Lecture 8.3.

In [ ]:
import json
from google.adk.tools import ToolContext

def sum_costs(costs: list[float]) -> float:
    """Calculates the sum of a list of numbers."""
    print(f"  [Tool Call] sum_costs on the list: {costs}")
    return sum(costs)

def exit_loop(tool_context: ToolContext):
    """Call this function ONLY when the plan is approved and within budget."""
    print(f"  [Tool Call] Budget approved. Terminating loop: {json.dumps(tool_context.state.to_dict())}")
    tool_context.actions.escalate = True
    return None

## 6. Create Tool Wrappers

Unchanged from Section 5 and Lecture 8.3.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool

google_search_agent = Agent(
    name="Google_Search_agent",
    model=AGENT_MODEL,
    instruction="You are just a wrapper for the Google Search tool.",
    tools=[google_search]
)

google_search_tool = AgentTool(agent=google_search_agent)

## 📝 7. Create Agents

All agents unchanged from Section 5.  

The guardrail lives exclusively on `budget_optimizer_workflow` in the next cell.

In [ ]:
COMPLETION_PHRASE = "The plan is within the budget."

# Agent 1: Proposes the initial, expensive plan (runs once).
# before_agent_callback=log_agent_entry carried over from 8.3 unchanged.
spending_proposer_agent = Agent(
    name="spending_proposer_agent",
    model=AGENT_MODEL,
    tools=[google_search],
    instruction="""
    You are a luxury event planner. For a {{topic}}, find a high-end venue and a gourmet catering service.

    Output a JSON object with items and their estimated costs, like:
    {"venue": {"name": "The Ritz London", "cost": 10000}, "catering": {"name": "Gourmet Chefs Inc.", "cost": 5000}}
    """,
    output_key="current_plan",
    before_agent_callback=log_agent_entry,   # ← 8.3 (unchanged)
)

# Agent 2 (in loop): The "Accountant" that critiques the plan.
# Unchanged from Section 5.
accountant_agent = Agent(
    name="accountant_agent",
    model=AGENT_MODEL,
    tools=[sum_costs],
    instruction=f"""
    You are a meticulous accountant. Your budget is {{{{budget}}}}.
    The current plan is: {{{{current_plan}}}}

    Extract the costs from the plan and use the `sum_costs` tool to get the total.
    - IF the total cost is > {{{{budget}}}}, output a critique like: "This plan is over budget by [amount]. Find a cheaper [item]."
    - ELSE, respond with the exact phrase: '{COMPLETION_PHRASE}'
    """,
    output_key="critique",
)

# Agent 3 (in loop): The "Cost Cutter" that refines the plan.
# Unchanged from Section 5.
cost_cutter_agent = Agent(
    name="cost_cutter_agent",
    model=AGENT_MODEL,
    tools=[google_search_tool, exit_loop],
    instruction=f"""
    You are a cost-cutting expert. You must refine a plan based on a critique.
    The critique is: {{{{critique}}}}
    The current plan is: {{{{current_plan}}}}

    - IF the critique is '{COMPLETION_PHRASE}'
        1. You MUST call the `exit_loop` tool with no arguments.
        2. After calling exit_loop, output the current plan EXACTLY as-is, character for character,
           with no modifications, no acknowledgements, no commentary, and no extra text. Do not summarize it.
           Do not rephrase it. Do not add "Budget approved" or any other text.
           Just echo {{{{current_plan}}}} verbatim.
    - ELSE, read the critique to identify the overpriced item. Use your search tool to find a cheaper alternative for that item.
      Output a new JSON object with the updated plan.
    """,
    output_key="current_plan",
)

# Agent 4: Presents the final approved plan (runs once after loop).
# Unchanged from 8.3.
plan_retriever_agent = Agent(
    name="plan_retriever_agent",
    model=AGENT_MODEL,
    instruction="""
    You are a plan finalizer. Your only job is to present the final, approved plan.
    The plan is available in the context variable `{{current_plan}}`.

    Your output must be the content of the final plan presented in a clear and easy-to-read format.
    """,
    tools=[],
    output_key="final_presentation",
    after_agent_callback=log_agent_exit,   # ← 8.3 (unchanged)
)


## 🔄 8. Assemble the Loop and Sequential Workflows

Unchanged from Section 5 and Lecture 8.3.

In [ ]:
from google.adk.agents import SequentialAgent, LoopAgent

# Unchanged from Section 5.
budget_refinement_loop = LoopAgent(
    name="budget_refinement_loop",
    sub_agents=[accountant_agent, cost_cutter_agent],
    max_iterations=3,
)

# ← 8.4 CHANGE: before_agent_callback=guardrail_before_workflow
# The guardrail lives here — on the SequentialAgent itself.
# This is the outermost boundary of the entire pipeline.
# When the judge blocks a topic, before_agent_callback returns Content,
# which skips _run_async_impl entirely — no sub-agent ever starts.
budget_optimizer_workflow = SequentialAgent(
    name="budget_optimizer_workflow",
    sub_agents=[spending_proposer_agent, budget_refinement_loop, plan_retriever_agent],
    # ← 8.4 CHANGE
)


## 🚀 9. Build the Execution Engine

Unchanged from Section 5 and Lecture 8.3.

In [ ]:
from IPython.display import display, Markdown

from google.adk.sessions import Session
from google.genai.types import Content, Part
from google.adk.runners import Runner

async def run_agent_query(agent: Agent, query: str, topic: str, budget: str, session: Session, user_id: str):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user"),
            state_delta={"budget": budget, "topic": topic, "COMPLETION_PHRASE": COMPLETION_PHRASE}
        ):
            pass
    except Exception as e:
        final_response = f"An error occurred: {e}"
        return final_response

    # Read the final response from session state.
    #
    # Two possible paths through the workflow:
    #
    #   ✅ CLEAN topic  → plan_retriever_agent runs and writes final_presentation.
    #                     We read that.
    #
    #   🚫 BLOCKED topic → guardrail cancels the workflow and writes refusal_reason
    #                      into state. plan_retriever never runs, so
    #                      final_presentation is never written. We read
    #                      refusal_reason instead.
    #
    final_session = await session_service.get_session(
        app_name=agent.name,
        user_id=user_id,
        session_id=session.id
    )
    state = final_session.state

    if "final_presentation" in state:
        final_response = state["final_presentation"]
    elif "refusal_reason" in state:
        final_response = state["refusal_reason"]
    else:
        final_response = "No response was generated."

    print("\n" + "-"*50)
    print("✅ Final Response:")
    display(Markdown(final_response))
    print("-"*50 + "\n")

    return final_response


## ✨ 10. Initialize Session Service

Unchanged from Section 5 and Lecture 8.3.

In [ ]:
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()
user_id = "adk_event_planner_001"

## ▶️ 11a. Run — CLEAN Topic (Guardrail Allows)

The topic `"50 person AI event in New York"` contains no unsafe intent.  
The safety judge will return `NO` and the full workflow runs as normal.

Watch for the **8.3 ENTRY log** from `log_agent_entry`, then the **SAFETY JUDGE** block  
confirming the topic is safe — then the workflow proceeds.

In [ ]:
async def run_clean_topic():
    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "50 person AI event in New York"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)

await run_clean_topic()

## 🚫 11b. Run — BLOCKED Topic (Guardrail Intercepts)

The topic `"weapons convention"` will be flagged as unsafe by the LLM judge.  
Watch for the `🚫 BLOCKED` line — **no sub-agent ever starts**.  
The final response is a tailored refusal explanation generated by the judge itself.

Notice what is absent from the output: no `[ENTRY]` log from `spending_proposer_agent`,  
no tool calls, no loop iterations, no `[EXIT]` log. The workflow is cancelled  
at the outermost boundary before any planning work begins.

In [ ]:
async def run_blocked_topic():
    session = await session_service.create_session(
        app_name=budget_optimizer_workflow.name,
        user_id=user_id
    )

    budget = 15000
    topic  = "weapons convention"
    query  = f"Find a plan for {topic}"

    print(f"User: {query}\n")
    await run_agent_query(budget_optimizer_workflow, query, topic, budget, session, user_id)

await run_blocked_topic()